# LeetCode 2913 — Subarrays Distinct Element Sum of Squares I bằng Python

Link đề: https://leetcode.com/problems/subarrays-distinct-element-sum-of-squares-i/ · Độ khó: **Easy** · Chủ đề: Array, Hash Set, Enumeration

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với **mọi mảng con liên tiếp** của `nums`, đếm số **giá trị khác nhau** (distinct count) trong nó, **bình phương** số đó, rồi **cộng tất cả**.
2. Phân biệt **mảng con liên tiếp (subarray)** với dãy con, biết có `n(n+1)/2` mảng con, và dùng `set` để đếm giá trị khác nhau.
3. Đi từ cách **`len(set(nums[i:j+1]))` cho từng mảng con `O(n³)`** → **mở rộng dần mảng con từ cố định đầu `i`, giữ một `set` `O(n²)`** → **bitmask số nguyên thay `set`** → **đếm cho mọi đầu `i` cùng lúc theo "lần xuất hiện cuối cùng"** (ý tưởng dẫn tới Bài II).
4. Hiểu **tại sao cập nhật tăng dần đúng**: khi thêm một phần tử mới, số giá trị khác nhau chỉ tăng 1 nếu phần tử đó **chưa có** trong mảng con.
5. Tránh các lỗi kinh điển: cộng `count` thay vì `count²`, đếm **tổng độ dài** thay vì giá trị khác nhau, reset `set` sai chỗ, và quên mảng con có độ dài `1`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh chỉ số từ `0`).

- **Mảng con (subarray)** là một dãy phần tử **liên tiếp, không rỗng** của `nums`.
- **Số giá trị khác nhau (distinct count)** của mảng con `nums[i..j]` là số giá trị **khác nhau** xuất hiện trong nó.

Hãy trả về **tổng bình phương** của distinct count của **mọi** mảng con của `nums`.

**Ví dụ 1**

- Input: `nums = [1, 2, 1]`
- Output: `15`
- Giải thích: có 6 mảng con:
  - `[1]`: 1 giá trị -> `1² = 1`
  - `[2]`: 1 -> `1`
  - `[1]` (cuối): 1 -> `1`
  - `[1, 2]`: 2 giá trị -> `4`
  - `[2, 1]`: 2 -> `4`
  - `[1, 2, 1]`: 2 giá trị khác nhau (`1` và `2`) -> `4`

  Tổng `1 + 1 + 1 + 4 + 4 + 4 = 15`.

**Ví dụ 2**

- Input: `nums = [1, 1]`
- Output: `3`
- Giải thích: 3 mảng con `[1]`, `[1]`, `[1, 1]` đều có đúng 1 giá trị khác nhau, nên tổng `1 + 1 + 1 = 3`.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random

examples = [
    ([1, 2, 1], 15),
    ([1, 1], 3),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 1] -> kỳ vọng 15
nums = [1, 1] -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Mảng con liên tiếp và cắt lát `nums[i:j+1]`

**Mảng con** là đoạn **sát nhau**. Trong Python lấy mảng con từ chỉ số `i` đến `j` (cả hai đầu) bằng **cắt lát** `nums[i:j+1]` (cận phải `j + 1` **không** được gồm). Cặp `(i, j)` với `i <= j` xác định đúng một mảng con.

In [2]:
nums = [1, 2, 1]
for i in range(len(nums)):
    for j in range(i, len(nums)):
        print((i, j), nums[i:j + 1])

(0, 0) [1]
(0, 1) [1, 2]
(0, 2) [1, 2, 1]
(1, 1) [2]
(1, 2) [2, 1]
(2, 2) [1]


### 3.2. Có bao nhiêu mảng con?

Với mảng dài `n`, số cặp `(i, j)` thỏa `0 <= i <= j < n` là `n + (n-1) + ... + 1 = n(n+1)/2`. Ví dụ `n = 3` có `6` mảng con; `n = 100` có `5050`. Con số này nhỏ nên ta có thể liệt kê hết khi `n <= 100`.

In [3]:
for n in [1, 2, 3, 10, 100]:
    count = sum(1 for i in range(n) for j in range(i, n))
    print(n, "->", count, "=", n * (n + 1) // 2)

1 -> 1 = 1
2 -> 3 = 3
3 -> 6 = 6
10 -> 55 = 55
100 -> 5050 = 5050


### 3.3. `set` và `len(set(...))`: đếm giá trị khác nhau

`set` là tập hợp **không chứa phần tử trùng**. Tạo `set` từ danh sách rồi `len(...)` cho **số giá trị khác nhau**: `len(set([1, 2, 1])) == 2`. Thêm phần tử: `s.add(x)` (nếu đã có thì không đổi). Kiểm tra: `x in s` (nhanh, trung bình `O(1)`).

In [4]:
s = set([1, 2, 1])
print(s, len(s))               # {1, 2} 2
s.add(2)                       # đã có: không đổi
s.add(5)
print(s, len(s))               # {1, 2, 5} 3
print(2 in s, 7 in s)          # True False

{1, 2} 2
{1, 2, 5} 3
True False


### 3.4. Bình phương và `**`

`x ** 2` hoặc `x * x` là bình phương. Đề lấy **bình phương của số giá trị khác nhau** rồi cộng, không phải cộng số giá trị khác nhau. Với `distinct = 3` thì thêm `9`, không phải `3`.

In [5]:
for d in [1, 2, 3, 4]:
    print(d, "->", d ** 2, d * d)
print("tổng bình phương của [1, 1, 2]:", sum(d ** 2 for d in [1, 1, 2]))     # 6
print("bình phương của tổng [1, 1, 2]:", sum([1, 1, 2]) ** 2)               # 16: khác

1 -> 1 1
2 -> 4 4
3 -> 9 9
4 -> 16 16
tổng bình phương của [1, 1, 2]: 6
bình phương của tổng [1, 1, 2]: 16


### 3.5. Cố định đầu `i` và mở rộng cuối `j` (đếm tăng dần)

Thay vì tạo `set` mới cho từng `(i, j)`, ta **cố định `i`** và cho `j` chạy sang phải, giữ **một `set` chung** và thêm `nums[j]` vào. Sau mỗi lần thêm, `len(set)` chính là distinct count của `nums[i..j]`. Cách này tránh làm lại việc, vì mảng con `nums[i..j+1]` chỉ hơn `nums[i..j]` đúng một phần tử.

In [6]:
nums = [1, 2, 1]
i = 0
seen = set()
for j in range(i, len(nums)):
    seen.add(nums[j])
    print(f"i={i}, j={j}: thêm {nums[j]} -> seen={sorted(seen)}, distinct={len(seen)}")

i=0, j=0: thêm 1 -> seen=[1], distinct=1
i=0, j=1: thêm 2 -> seen=[1, 2], distinct=2
i=0, j=2: thêm 1 -> seen=[1, 2], distinct=2


### 3.6. Bitmask: dùng một số nguyên làm tập hợp

Vì giá trị chỉ từ `1` đến `100`, ta có thể biểu diễn một tập hợp bằng **một số nguyên** `mask`: bit thứ `x` bằng `1` nghĩa là "đã thấy giá trị `x`". Thêm `x`: `mask |= 1 << x` (`|` là OR bit, `1 << x` là số có đúng bit `x`). Đếm số giá trị khác nhau: `mask.bit_count()` (số bit `1`). Python hỗ trợ số nguyên dài tùy ý nên `1 << 100` không tràn.

In [7]:
mask = 0
for x in [1, 2, 1]:
    mask |= 1 << x
    print(f"thêm {x}: mask={bin(mask)} -> distinct={mask.bit_count()}")

thêm 1: mask=0b10 -> distinct=1
thêm 2: mask=0b110 -> distinct=2
thêm 1: mask=0b110 -> distinct=2


### 3.7. `dict` ghi "lần xuất hiện cuối cùng" của mỗi giá trị

`last = {}`; sau khi xử lý vị trí `j` ta ghi `last[nums[j]] = j`. Tra cứu `last.get(x, -1)` cho **vị trí xuất hiện gần nhất trước đó** (hoặc `-1` nếu chưa từng xuất hiện). Ta sẽ dùng nó ở Cách 4 để biết "thêm `nums[j]` làm distinct count của những mảng con nào tăng thêm 1".

In [8]:
nums = [1, 2, 1]
last = {}
for j, x in enumerate(nums):
    print(f"j={j}, x={x}: lần trước xuất hiện ở {last.get(x, -1)}")
    last[x] = j

j=0, x=1: lần trước xuất hiện ở -1
j=1, x=2: lần trước xuất hiện ở -1
j=2, x=1: lần trước xuất hiện ở 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy liệt kê **mọi mảng con** theo đầu `i` và cuối `j`, mỗi mảng con ghi số giá trị khác nhau rồi bình phương. Với `nums = [1, 2, 1]`:

| `(i, j)` | mảng con | giá trị khác nhau | distinct | bình phương |
|---|---|---|---|---|
| (0, 0) | `[1]` | {1} | 1 | 1 |
| (0, 1) | `[1, 2]` | {1, 2} | 2 | 4 |
| (0, 2) | `[1, 2, 1]` | {1, 2} | 2 | 4 |
| (1, 1) | `[2]` | {2} | 1 | 1 |
| (1, 2) | `[2, 1]` | {2, 1} | 2 | 4 |
| (2, 2) | `[1]` | {1} | 1 | 1 |

Tổng `1 + 4 + 4 + 1 + 4 + 1 = 15`. Chú ý `[1, 2, 1]` chỉ có **2** giá trị khác nhau dù dài 3, vì số `1` xuất hiện hai lần. Thuật toán: **duyệt mọi mảng con, đếm distinct, bình phương, cộng dồn**.

## 5. Cách 1 — Brute Force / baseline: `len(set(nums[i:j+1]))` cho từng mảng con

**Ý tưởng.** Hai vòng lặp `(i, j)` chọn mảng con, tạo `set` từ lát cắt rồi lấy độ dài và bình phương. Cách trực tiếp nhất.

In [9]:
def sum_counts_brute(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for i in range(n):
        for j in range(i, n):
            distinct = len(set(nums[i:j + 1]))      # tạo set mới từ lát cắt
            total += distinct * distinct
    return total


for nums, expected in examples:
    print(nums, "->", sum_counts_brute(nums), "(kỳ vọng", expected, ")")

[1, 2, 1] -> 15 (kỳ vọng 15 )
[1, 1] -> 3 (kỳ vọng 3 )


In [10]:
# Từng mảng con và distinct của ví dụ 1
nums = [1, 2, 1]
total = 0
for i in range(len(nums)):
    for j in range(i, len(nums)):
        d = len(set(nums[i:j + 1]))
        total += d * d
        print(f"({i},{j}) {nums[i:j + 1]} distinct={d} -> cộng {d * d}, total={total}")

(0,0) [1] distinct=1 -> cộng 1, total=1
(0,1) [1, 2] distinct=2 -> cộng 4, total=5
(0,2) [1, 2, 1] distinct=2 -> cộng 4, total=9
(1,1) [2] distinct=1 -> cộng 1, total=10
(1,2) [2, 1] distinct=2 -> cộng 4, total=14
(2,2) [1] distinct=1 -> cộng 1, total=15


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài `nums`. Có `O(n²)` mảng con, mỗi lần tạo `set` từ lát cắt dài tới `n` tốn `O(n)`: **Time `O(n³)`**; lát cắt và `set` tạm tối đa `n` phần tử: **Space `O(n)`**. Với `n <= 100` khoảng `100³/6 ≈ 170 000` phép, vẫn chạy kịp.

**Nút thắt.** Mỗi mảng con **làm lại từ đầu** việc đếm, trong khi `nums[i..j+1]` chỉ hơn `nums[i..j]` đúng một phần tử. Ta có thể **tái sử dụng** `set` của mảng con trước.

## 6. Ý tưởng thuật toán / bước đột phá

**Mở rộng dần.** Với mỗi đầu `i`, cho `j` chạy từ `i` sang phải và **thêm `nums[j]` vào một `set` chung**. `len(set)` sau mỗi lần thêm chính là distinct count của `nums[i..j]`. Mỗi `(i, j)` chỉ tốn `O(1)` (thêm một phần tử), nên tổng `O(n²)`.

**Vì sao đúng.** `nums[i..j]` gồm các phần tử của `nums[i..j-1]` thêm `nums[j]`. Tập giá trị của nó là hợp của tập trước và `{nums[j]}`; kích thước tăng 1 nếu `nums[j]` chưa có, ngược lại giữ nguyên. Đó đúng là hành vi của `set.add`.

## 7. Các cách tiếp theo

### Cách 2 — Cố định `i`, mở rộng `j` với một `set` chung `O(n²)`

**Giải thích.** Với mỗi `i`, tạo `set` rỗng; vòng `j` thêm `nums[j]` rồi cộng `len(seen)²` vào đáp án.

In [11]:
def sum_counts_incremental(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for i in range(n):
        seen = set()                           # set mới cho mỗi đầu i
        for j in range(i, n):
            seen.add(nums[j])                  # nums[i..j] = nums[i..j-1] + nums[j]
            total += len(seen) ** 2
    return total


for nums, expected in examples:
    print(nums, "->", sum_counts_incremental(nums), "(kỳ vọng", expected, ")")

[1, 2, 1] -> 15 (kỳ vọng 15 )
[1, 1] -> 3 (kỳ vọng 3 )


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List

class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        n = len(nums)
        total = 0
        for i in range(n):
            seen = set()
            for j in range(i, n):
                seen.add(nums[j])
                total += len(seen) ** 2
        return total

**Độ phức tạp Cách 2:** **Time `O(n²)`** (mỗi `(i, j)` một lần `add` và `len`), **Space `O(n)`** (một `set` tối đa `n` phần tử).

### Cách 3 — Bitmask số nguyên thay cho `set`

**Giải thích.** Vì giá trị `1..100`, biểu diễn "tập đã thấy" bằng số nguyên `mask`; thêm `x` bằng `mask |= 1 << x`; distinct bằng `mask.bit_count()`. Cùng độ phức tạp `O(n²)` nhưng hằng số nhỏ (không tạo đối tượng `set`).

In [13]:
def sum_counts_bitmask(nums: List[int]) -> int:
    n = len(nums)
    total = 0
    for i in range(n):
        mask = 0
        for j in range(i, n):
            mask |= 1 << nums[j]               # bật bit nums[j]
            d = mask.bit_count()               # số bit 1 = số giá trị khác nhau
            total += d * d
    return total


for nums, expected in examples:
    print(nums, "->", sum_counts_bitmask(nums), "(kỳ vọng", expected, ")")

[1, 2, 1] -> 15 (kỳ vọng 15 )
[1, 1] -> 3 (kỳ vọng 3 )


**Độ phức tạp Cách 3:** **Time `O(n²)`** (mỗi bước là phép bit trên số nguyên 100 bit, coi như hằng số), **Space `O(1)`** (một số nguyên). Ghi chú: `bit_count()` cần Python 3.10+; bản cũ dùng `bin(mask).count("1")`.

### Cách 4 — Đếm cho mọi đầu `i` cùng lúc, theo lần xuất hiện cuối cùng

**Giải thích.** Gọi `cnt[i]` là distinct count của `nums[i..j]` (cuối cố định là `j`, đầu `i` thay đổi). Khi thêm phần tử `x = nums[j]`, ta chuyển từ `j - 1` sang `j`:

- tạo thêm mảng con `nums[j..j]` với `cnt[j] = 0` rồi **cộng 1**;
- gọi `p` là chỉ số xuất hiện **cuối cùng trước đó** của `x` (`-1` nếu chưa có). Với các đầu `i` thuộc `(p, j]`, mảng con `nums[i..j-1]` **chưa chứa `x`** (vì `x` lần trước ở `p < i`), nên distinct **tăng thêm 1**. Với `i <= p`, `x` đã có sẵn, distinct **không đổi**.

Sau đó `total += Σ cnt[i]²` cho mọi `i <= j` (đây là mọi mảng con kết thúc ở `j`). Đây là ý tưởng "cập nhật khoảng `(p, j]`" mà Bài II (với `n` tới `10^5`) tăng tốc bằng cây phân đoạn (segment tree).

In [14]:
def sum_counts_last_occurrence(nums: List[int]) -> int:
    n = len(nums)
    cnt = [0] * n                              # cnt[i] = distinct của nums[i..j] (cuối là j hiện tại)
    last = {}                                  # giá trị -> vị trí xuất hiện cuối cùng
    total = 0
    for j, x in enumerate(nums):
        p = last.get(x, -1)                    # vị trí lần trước (hoặc -1)
        for i in range(p + 1, j + 1):          # các đầu i trong (p, j]
            cnt[i] += 1
        last[x] = j
        total += sum(c * c for c in cnt[:j + 1])    # mọi mảng con kết thúc tại j
    return total


for nums, expected in examples:
    print(nums, "->", sum_counts_last_occurrence(nums), "(kỳ vọng", expected, ")")

[1, 2, 1] -> 15 (kỳ vọng 15 )
[1, 1] -> 3 (kỳ vọng 3 )


In [15]:
# Theo dõi cnt qua từng j với ví dụ 1: [1, 2, 1]
nums = [1, 2, 1]
cnt, last, total = [0] * 3, {}, 0
for j, x in enumerate(nums):
    p = last.get(x, -1)
    for i in range(p + 1, j + 1):
        cnt[i] += 1
    last[x] = j
    s = sum(c * c for c in cnt[:j + 1])
    total += s
    print(f"j={j}, x={x}, p={p}: cộng 1 cho i in ({p}, {j}] -> cnt[:j+1]={cnt[:j + 1]}, bình phương tổng={s}, total={total}")

j=0, x=1, p=-1: cộng 1 cho i in (-1, 0] -> cnt[:j+1]=[1], bình phương tổng=1, total=1
j=1, x=2, p=-1: cộng 1 cho i in (-1, 1] -> cnt[:j+1]=[2, 1], bình phương tổng=5, total=6
j=2, x=1, p=0: cộng 1 cho i in (0, 2] -> cnt[:j+1]=[2, 2, 1], bình phương tổng=9, total=15


**Độ phức tạp Cách 4:** **Time `O(n²)`** (mỗi `j` cập nhật tối đa `j + 1` ô và tính tổng bình phương `j + 1` ô), **Space `O(n)`** (`cnt`, `last`). Cách này **không nhanh hơn** Cách 2 ở Bài I nhưng đặt nền cho Bài II.

> Cách 2 là cách nên nộp (ngắn nhất, rõ nhất). Cách 3 nhanh hơn một chút; Cách 4 đáng học vì tư duy "cập nhật khoảng khi thêm một phần tử".

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với `nums = [1, 2, 1]`; (b) Cách 4 với `nums = [3, 1, 3, 2]` có giá trị lặp.

In [16]:
def trace_incremental(nums):
    n, total = len(nums), 0
    print("nums =", nums)
    for i in range(n):
        seen = set()
        print(f"--- đầu i={i}: seen = {{}}")
        for j in range(i, n):
            seen.add(nums[j])
            d = len(seen)
            total += d * d
            print(f"  j={j}: thêm {nums[j]} -> seen={sorted(seen)} distinct={d} -> cộng {d * d}, total={total}")
    print("=> tổng =", total)
    return total


assert trace_incremental([1, 2, 1]) == 15

nums = [1, 2, 1]
--- đầu i=0: seen = {}
  j=0: thêm 1 -> seen=[1] distinct=1 -> cộng 1, total=1
  j=1: thêm 2 -> seen=[1, 2] distinct=2 -> cộng 4, total=5
  j=2: thêm 1 -> seen=[1, 2] distinct=2 -> cộng 4, total=9
--- đầu i=1: seen = {}
  j=1: thêm 2 -> seen=[2] distinct=1 -> cộng 1, total=10
  j=2: thêm 1 -> seen=[1, 2] distinct=2 -> cộng 4, total=14
--- đầu i=2: seen = {}
  j=2: thêm 1 -> seen=[1] distinct=1 -> cộng 1, total=15
=> tổng = 15


In [17]:
def trace_last_occurrence(nums):
    n = len(nums)
    cnt, last, total = [0] * n, {}, 0
    print("nums =", nums)
    for j, x in enumerate(nums):
        p = last.get(x, -1)
        for i in range(p + 1, j + 1):
            cnt[i] += 1
        last[x] = j
        s = sum(c * c for c in cnt[:j + 1])
        total += s
        print(f"j={j} x={x} p={p}: cộng 1 cho i trong ({p}, {j}] -> cnt={cnt[:j + 1]} | Σ cnt² = {s} | total={total}")
    return total


assert trace_last_occurrence([3, 1, 3, 2]) == sum_counts_brute([3, 1, 3, 2])
print("tổng khớp brute force:", sum_counts_brute([3, 1, 3, 2]))

nums = [3, 1, 3, 2]
j=0 x=3 p=-1: cộng 1 cho i trong (-1, 0] -> cnt=[1] | Σ cnt² = 1 | total=1
j=1 x=1 p=-1: cộng 1 cho i trong (-1, 1] -> cnt=[2, 1] | Σ cnt² = 5 | total=6
j=2 x=3 p=0: cộng 1 cho i trong (0, 2] -> cnt=[2, 2, 1] | Σ cnt² = 9 | total=15
j=3 x=2 p=-1: cộng 1 cho i trong (-1, 3] -> cnt=[3, 3, 2, 1] | Σ cnt² = 23 | total=38
tổng khớp brute force: 38


Bảng tóm tắt (Cách 4, `nums = [3, 1, 3, 2]`):

| `j` | `x` | `p` (lần trước) | `cnt[:j+1]` sau cập nhật | `Σ cnt²` |
|---|---|---|---|---|
| 0 | 3 | -1 | `[1]` | 1 |
| 1 | 1 | -1 | `[2, 1]` | 5 |
| 2 | 3 | 0 | `[2, 2, 1]` | 9 |
| 3 | 2 | -1 | `[3, 3, 2, 1]` | 23 |

Tổng `1 + 5 + 9 + 23 = 38`. (Ở `j = 2`, `x = 3` đã xuất hiện ở `p = 0` nên chỉ các đầu `i = 1, 2` được cộng 1; `cnt[0]` giữ nguyên.)

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (set mở rộng dần). Đúng class `Solution`, đúng method `sumCounts(self, nums: List[int]) -> int`.

In [18]:
from typing import List

class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        n = len(nums)
        total = 0
        for i in range(n):
            seen = set()                       # giá trị khác nhau của nums[i..j]
            for j in range(i, n):
                seen.add(nums[j])
                total += len(seen) ** 2
        return total


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.sumCounts(nums))

[1, 2, 1] -> 15
[1, 1] -> 3


## 10. Vì sao thuật toán đúng

- **Mọi mảng con được xét đúng một lần:** mỗi mảng con ứng với một cặp `(i, j)` với `i <= j`; hai vòng lặp duyệt đúng các cặp đó.
- **`seen` đúng bằng tập giá trị của `nums[i..j]`:** ở `j = i` `seen = {nums[i]}`. Giả sử đúng cho `j - 1`, thì thêm `nums[j]` cho tập của `nums[i..j]`. `set` không lưu trùng nên `len(seen)` là số giá trị khác nhau.
- **Bình phương rồi cộng:** mỗi mảng con đóng góp `distinct²`, tổng chính là đáp án.
- **Cách 3 và Cách 4 tương đương:** bitmask là một biểu diễn khác của cùng tập; Cách 4 duy trì distinct của mọi đầu `i` bằng quy tắc "thêm `x` làm tăng distinct của các đầu `i > p` (với `p` là vị trí lần trước của `x`)", đúng vì mảng con `nums[i..j-1]` chứa `x` khi và chỉ khi `i <= p`.
- **Tính chất kiểm được:** đáp án nằm giữa `n(n+1)/2` (mọi mảng con đều distinct = 1) và tổng `Σ len²` của mọi mảng con (mọi phần tử khác nhau).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài `nums`.

- **Cách 1:** Time `O(n³)`, Space `O(n)`.
- **Cách 2:** Time **`O(n²)`**, Space `O(n)`.
- **Cách 3:** Time `O(n²)` (hằng số nhỏ), Space `O(1)`.
- **Cách 4:** Time `O(n²)`, Space `O(n)`.
- Có `n(n+1)/2` mảng con nên mọi cách "liệt kê từng mảng con" đều `Ω(n²)`. Bài II (`n` tới `10^5`) cần `O(n log n)` bằng cây phân đoạn với cập nhật khoảng và truy vấn tổng bình phương, nên không còn liệt kê từng mảng con được.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [19]:
sol = Solution()
impls = {
    "brute": sum_counts_brute,
    "incremental": sum_counts_incremental,
    "bitmask": sum_counts_bitmask,
    "last_occurrence": sum_counts_last_occurrence,
    "submit": sol.sumCounts,
}

for nums, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, nums, got, expected)
    print("PASS", label, "->", expected)


check_all([5], 1, "một phần tử: một mảng con, distinct = 1")
check_all([1, 1, 1], 6, "toàn giống nhau: 6 mảng con, mỗi mảng distinct = 1")
check_all([1, 2], 1 + 1 + 4, "hai giá trị khác nhau: 1 + 1 + 2²")
check_all([1, 2, 3], 3 * 1 + 2 * 4 + 1 * 9, "mọi phần tử khác nhau: Σ len² qua mọi mảng con")
check_all([100, 100, 1], 1 + 1 + 1 + 1 + 4 + 4, "giá trị lớn 100 và lặp")
check_all([1, 2, 1, 2], 1 + 1 + 1 + 1 + 4 + 4 + 4 + 4 + 4 + 4, "xen kẽ hai giá trị")

PASS một phần tử: một mảng con, distinct = 1 -> 1
PASS toàn giống nhau: 6 mảng con, mỗi mảng distinct = 1 -> 6
PASS hai giá trị khác nhau: 1 + 1 + 2² -> 6
PASS mọi phần tử khác nhau: Σ len² qua mọi mảng con -> 20
PASS giá trị lớn 100 và lặp -> 12
PASS xen kẽ hai giá trị -> 28


### 12.2. Tính chất và phản ví dụ

In [21]:
# Tính chất 1: cận dưới n(n+1)/2 và cận trên Σ len² qua mọi mảng con
random.seed(7)
for _ in range(200):
    n = random.randint(1, 20)
    nums = [random.randint(1, 100) for _ in range(n)]
    r = sol.sumCounts(nums)
    lo = n * (n + 1) // 2
    hi = sum((j - i + 1) ** 2 for i in range(n) for j in range(i, n))
    assert lo <= r <= hi
print("PASS n(n+1)/2 <= đáp án <= Σ len² qua mọi mảng con")

# Tính chất 2: mọi phần tử khác nhau thì đáp án đúng bằng cận trên
n = 15
distinct = list(range(1, n + 1))
assert sol.sumCounts(distinct) == sum((j - i + 1) ** 2 for i in range(n) for j in range(i, n))
# và mọi phần tử bằng nhau thì bằng cận dưới
assert sol.sumCounts([7] * n) == n * (n + 1) // 2
print("PASS mọi phần tử khác nhau -> cận trên; mọi phần tử bằng nhau -> cận dưới")

# Tính chất 3: đổi tên giá trị (song ánh) không đổi đáp án
nums = [random.randint(1, 5) for _ in range(30)]
mapping = {v: 100 - v for v in range(1, 6)}
assert sol.sumCounts(nums) == sol.sumCounts([mapping[v] for v in nums])
print("PASS đổi nhãn giá trị không đổi đáp án")

# Đầu vào không bị sửa
nums = [1, 2, 1, 3]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: cộng distinct thay vì distinct²
def sai_khong_binh_phuong(nums):
    return sum(len(set(nums[i:j + 1])) for i in range(len(nums)) for j in range(i, len(nums)))

assert sai_khong_binh_phuong([1, 2, 1]) == 9 != 15
print("PASS phản ví dụ: quên bình phương cho", sai_khong_binh_phuong([1, 2, 1]), "thay vì 15")

# Phản ví dụ 2: dùng độ dài mảng con thay vì số giá trị khác nhau
def sai_dung_do_dai(nums):
    return sum((j - i + 1) ** 2 for i in range(len(nums)) for j in range(i, len(nums)))

assert sai_dung_do_dai([1, 2, 1]) == 1 + 1 + 1 + 4 + 4 + 9
print("PASS phản ví dụ: dùng độ dài cho", sai_dung_do_dai([1, 2, 1]), "thay vì 15")

PASS n(n+1)/2 <= đáp án <= Σ len² qua mọi mảng con
PASS mọi phần tử khác nhau -> cận trên; mọi phần tử bằng nhau -> cận dưới
PASS đổi nhãn giá trị không đổi đáp án
PASS đầu vào không bị sửa
PASS phản ví dụ: quên bình phương cho 9 thay vì 15
PASS phản ví dụ: dùng độ dài cho 20 thay vì 15


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [22]:
random.seed(2913)
for trial in range(500):
    n = random.randint(1, 40)
    hi = random.choice([2, 5, 20, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    ref = sum_counts_brute(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums, fn(list(nums)), ref)
print("PASS 500 mảng ngẫu nhiên (n tới 40) - mọi cách khớp nhau")

# Thử kích thước tối đa của đề (n = 100) cho các cách O(n^2)
nums = [random.randint(1, 100) for _ in range(100)]
ref = sum_counts_incremental(nums)
for key in ("bitmask", "last_occurrence", "submit", "brute"):
    assert impls[key](list(nums)) == ref, key
print("PASS n = 100: các cách khớp nhau, đáp án =", ref)

PASS 500 mảng ngẫu nhiên (n tới 40) - mọi cách khớp nhau
PASS n = 100: các cách khớp nhau, đáp án = 4731317


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cộng số giá trị khác nhau thay vì bình phương

In [23]:
def sai_khong_bp(nums):
    total = 0
    for i in range(len(nums)):
        seen = set()
        for j in range(i, len(nums)):
            seen.add(nums[j])
            total += len(seen)                    # thiếu bình phương
    return total


print("sai  :", sai_khong_bp([1, 2, 1]), "| đúng:", Solution().sumCounts([1, 2, 1]))

sai  : 9 | đúng: 15


**Vì sao sai:** đề yêu cầu **tổng của bình phương**. `[1, 2]` có distinct `2` phải đóng góp `4`, không phải `2`. Đây là lỗi hay gặp khi đọc đề lướt.

### Lỗi 2: Đếm độ dài mảng con thay vì số giá trị khác nhau

In [24]:
def sai_do_dai(nums):
    total = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            total += (j - i + 1) ** 2             # độ dài, không phải số giá trị khác nhau
    return total


print("sai  :", sai_do_dai([1, 1]), "| đúng:", Solution().sumCounts([1, 1]))

sai  : 6 | đúng: 3


**Vì sao sai:** distinct count đếm các **giá trị khác nhau**, không phải số phần tử. `[1, 1]` dài `2` nhưng chỉ có `1` giá trị khác nhau, nên đóng góp `1`, không phải `4`.

### Lỗi 3: Dùng chung một `set` cho mọi đầu `i` (quên khởi tạo lại)

In [25]:
def sai_khong_reset(nums):
    seen = set()                                   # đặt ngoài vòng i: không bao giờ xóa
    total = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            seen.add(nums[j])
            total += len(seen) ** 2
    return total


print("sai  :", sai_khong_reset([1, 2, 1]), "| đúng:", Solution().sumCounts([1, 2, 1]))

sai  : 21 | đúng: 15


**Vì sao sai:** mỗi **đầu `i` mới** phải bắt đầu với `set` **rỗng** (mảng con mới không liên quan phần tử trước `i`). Nếu không khởi tạo lại, các giá trị của các lần `i` trước còn nằm trong `set` và làm distinct bị phình lên.

### Lỗi 4: Bỏ sót mảng con độ dài 1 hoặc dùng sai cận lát cắt

In [26]:
def sai_can_lat_cat(nums):
    n, total = len(nums), 0
    for i in range(n):
        for j in range(i + 1, n):                  # bỏ mảng con độ dài 1 (j bắt đầu từ i + 1)
            total += len(set(nums[i:j + 1])) ** 2
    return total


def sai_cat_thieu(nums):
    n, total = len(nums), 0
    for i in range(n):
        for j in range(i, n):
            total += len(set(nums[i:j])) ** 2      # thiếu +1: lát cắt không gồm nums[j]
    return total


print("sai (bỏ độ dài 1):", sai_can_lat_cat([1, 2, 1]), "| đúng:", Solution().sumCounts([1, 2, 1]))
print("sai (nums[i:j])   :", sai_cat_thieu([1, 2, 1]))

sai (bỏ độ dài 1): 12 | đúng: 15
sai (nums[i:j])   : 6


**Vì sao sai:** mảng con `nums[i..j]` gồm cả `nums[j]`, nên lát cắt là `nums[i:j+1]`. Và `j` phải bắt đầu từ `i` (mảng con độ dài `1` cũng đóng góp `1`). `nums[i:j]` tạo mảng con **rỗng** khi `j = i` (distinct `0`) và thiếu phần tử cuối.

### Lỗi 5: Nhầm "mảng con" với "dãy con"

In [27]:
from itertools import combinations

def sai_day_con(nums):
    total = 0
    for r in range(1, len(nums) + 1):
        for idx in combinations(range(len(nums)), r):         # mọi dãy con (bỏ qua phần tử được)
            total += len(set(nums[i] for i in idx)) ** 2
    return total


print("sai  :", sai_day_con([1, 2, 1]), "| đúng:", Solution().sumCounts([1, 2, 1]))

sai  : 16 | đúng: 15


**Vì sao sai:** **mảng con** là đoạn **liên tiếp**, còn **dãy con** được bỏ phần tử. Với `n = 3` có `6` mảng con nhưng `7` dãy con không rỗng, và số dãy con tăng theo `2ⁿ`, rất nhanh. Đề bài chỉ xét mảng con (đoạn liên tiếp).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "mọi mảng con" với `n <= 100` → cho phép `O(n²)` hoặc thậm chí `O(n³)`; liệt kê `(i, j)` là ổn.
- Cố định đầu `i`, **mở rộng `j` và giữ cấu trúc tăng dần** (`set`, đếm...) thay vì tính lại từ đầu.
- "Số giá trị khác nhau" → `set` (hoặc bitmask nếu miền nhỏ).
- Đề ghi `n` lớn (Bài II) → cần ý tưởng **lần xuất hiện cuối cùng** + cấu trúc dữ liệu cho cập nhật khoảng.
- Kiểm tra nhanh: mảng toàn giống nhau cho `n(n+1)/2`; mảng toàn khác nhau cho `Σ len²`.
- Luôn tự đối chiếu bằng brute force trên dữ liệu ngẫu nhiên nhỏ.

## 15. Biến thể của bài

1. **Bài II** (cùng series): `n` lên tới `10^5`; cần cây phân đoạn (segment tree) với cập nhật khoảng cộng 1 và duy trì tổng bình phương.
2. Tổng **distinct count** (không bình phương) của mọi mảng con: dùng đóng góp từng phần tử `(i - p)·(n - i)` với `p` là vị trí lần trước.
3. Đếm số mảng con có **đúng `k`** giá trị khác nhau (cửa sổ trượt).
4. Mảng con dài nhất có tối đa `k` giá trị khác nhau.
5. Tổng **lũy thừa 3** của distinct count.
6. Cùng bài nhưng với **mảng chuỗi** (dùng `set` chuỗi).

Ta thử biến thể 2 (tổng distinct bằng công thức đóng góp) và 3 (đếm mảng con có đúng `k` giá trị):

In [28]:
def sum_distinct_counts(nums):
    """Biến thể 2: O(n) bằng đóng góp. Phần tử ở i đóng góp cho mọi mảng con [l, r] mà nó là lần xuất hiện ĐẦU tiên của giá trị trong mảng con đó."""
    n, last, total = len(nums), {}, 0
    for i, x in enumerate(nums):
        p = last.get(x, -1)
        total += (i - p) * (n - i)          # l thuộc (p, i], r thuộc [i, n)
        last[x] = i
    return total


def at_most_k(nums, k):
    """Số mảng con có tối đa k giá trị khác nhau (cửa sổ trượt)."""
    from collections import Counter
    cnt, left, res = Counter(), 0, 0
    for right, x in enumerate(nums):
        cnt[x] += 1
        while len(cnt) > k:
            cnt[nums[left]] -= 1
            if cnt[nums[left]] == 0:
                del cnt[nums[left]]
            left += 1
        res += right - left + 1
    return res


for _ in range(100):
    nums = [random.randint(1, 4) for _ in range(random.randint(1, 15))]
    brute = sum(len(set(nums[i:j + 1])) for i in range(len(nums)) for j in range(i, len(nums)))
    assert sum_distinct_counts(nums) == brute
print("2. tổng distinct của [1, 2, 1]:", sum_distinct_counts([1, 2, 1]))        # 1+1+1+2+2+2 = 9
exactly_k = lambda nums, k: at_most_k(nums, k) - at_most_k(nums, k - 1)
print("3. số mảng con có đúng 2 giá trị của [1, 2, 1, 2, 3]:", exactly_k([1, 2, 1, 2, 3], 2))

2. tổng distinct của [1, 2, 1]: 9
3. số mảng con có đúng 2 giá trị của [1, 2, 1, 2, 3]: 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `len(set(nums[i:j+1]))` mọi `(i, j)` | tạo set mới cho từng mảng con | `O(n³)` | `O(n)` | brute force để đối chiếu |
| 2. Cố định `i`, mở rộng `j`, một `set` | thêm dần 1 phần tử | `O(n²)` | `O(n)` | **cách nộp bài** |
| 3. Bitmask số nguyên | `mask \|= 1 << x`, `bit_count` | `O(n²)` (hằng số nhỏ) | `O(1)` | miền giá trị nhỏ (<= vài trăm) |
| 4. `cnt[i]` theo lần xuất hiện cuối | cập nhật khoảng `(p, j]` | `O(n²)` | `O(n)` | nền tảng cho Bài II (segment tree) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len(set([3, 3, 4]))`, `len([3, 3, 4][0:2])`, `Solution().sumCounts([5])`, `Solution().sumCounts([2, 2, 2])`, `(1 << 3) | (1 << 5)`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 2]`: liệt kê 6 mảng con, distinct count và bình phương của từng mảng con rồi cộng.

**Bài 3 (điền TODO).** Hoàn thành `sum_counts_todo(nums)` theo Cách 2.

In [29]:
def sum_counts_todo(nums):
    n = len(nums)
    total = 0
    for i in range(n):
        # TODO 1: tạo set rỗng seen
        for j in range(i, n):
            # TODO 2: thêm nums[j] vào seen
            # TODO 3: cộng len(seen) ** 2 vào total
            pass
    return total

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `sum_counts_bitmask2(nums)` dùng bitmask (Cách 3) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sum_distinct(nums)` trả về **tổng distinct count** (không bình phương) của mọi mảng con trong `O(n)` bằng công thức đóng góp `(i - p)·(n - i)`. (b) Viết `count_subarrays_all_distinct(nums)` đếm số mảng con mà **mọi phần tử khác nhau** (distinct count bằng độ dài), dùng cửa sổ trượt.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `2`, `1`, `6` (có 6 mảng con, mỗi mảng distinct 1), `40` (`8 + 32`). Bài 2: viết các cặp `(i, j)`.
- **Mức 2 (vừa):** Bài 2: `[1]`:1, `[2]`:1, `[2]`:1, `[1,2]`:4, `[2,2]`:1, `[1,2,2]`:4 -> tổng `12`. Bài 3: ba TODO đã ghi trong comment. Bài 4: `mask = 0`, `mask |= 1 << x`, `mask.bit_count()`.
- **Mức 3 (sâu):** Bài 5(a): với mỗi `i`, `p` = lần trước; số mảng con mà `i` là lần xuất hiện đầu tiên của giá trị là `(i - p)` lựa chọn đầu `l` nhân `(n - i)` lựa chọn cuối `r`. Bài 5(b): giữ cửa sổ `[left, right]` không trùng; mỗi `right` thêm `right - left + 1` mảng con.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert len(set([3, 3, 4])) == 2 and len([3, 3, 4][0:2]) == 2
assert Solution().sumCounts([5]) == 1
assert Solution().sumCounts([2, 2, 2]) == 6
assert (1 << 3) | (1 << 5) == 40

# Bài 2
assert Solution().sumCounts([1, 2, 2]) == 12

# Bài 3
def sum_counts_todo_ans(nums):
    n = len(nums)
    total = 0
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(nums[j])
            total += len(seen) ** 2
    return total

for nums, expected in examples:
    assert sum_counts_todo_ans(nums) == expected

# Bài 4
def sum_counts_bitmask2(nums):
    total = 0
    for i in range(len(nums)):
        mask = 0
        for j in range(i, len(nums)):
            mask |= 1 << nums[j]
            total += mask.bit_count() ** 2
    return total

for nums, expected in examples:
    assert sum_counts_bitmask2(nums) == expected

# Bài 5(a)
def sum_distinct(nums):
    n, last, total = len(nums), {}, 0
    for i, x in enumerate(nums):
        p = last.get(x, -1)
        total += (i - p) * (n - i)
        last[x] = i
    return total

for _ in range(200):
    nums = [random.randint(1, 5) for _ in range(random.randint(1, 15))]
    brute = sum(len(set(nums[i:j + 1])) for i in range(len(nums)) for j in range(i, len(nums)))
    assert sum_distinct(nums) == brute

# Bài 5(b)
def count_subarrays_all_distinct(nums):
    seen, left, res = set(), 0, 0
    for right, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[left])
            left += 1
        seen.add(x)
        res += right - left + 1
    return res

assert count_subarrays_all_distinct([1, 2, 1]) == 5      # [1],[2],[1],[1,2],[2,1]
assert count_subarrays_all_distinct([7, 7, 7]) == 3
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cố định đầu, mở rộng cuối, giữ một `set`)

```python
from typing import List

class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        n, total = len(nums), 0
        for i in range(n):
            seen = set()
            for j in range(i, n):
                seen.add(nums[j])
                total += len(seen) ** 2
        return total
```

**Checklist**

- [ ] Cộng **bình phương** của distinct count (không phải distinct, không phải độ dài)?
- [ ] `set` được **khởi tạo lại** cho mỗi đầu `i`?
- [ ] `j` chạy từ `i` (mảng con độ dài `1` cũng được tính); lát cắt `nums[i:j+1]` nếu dùng brute force?
- [ ] Hiểu "mảng con" là đoạn **liên tiếp** (không phải dãy con)?
- [ ] Độ phức tạp `O(n²)` phù hợp `n <= 100`; biết Bài II cần cách khác?
- [ ] Đã thử: một phần tử, toàn giống nhau, toàn khác nhau?
- [ ] Skeleton đúng `class Solution` / `def sumCounts(self, nums: List[int]) -> int`?